<a href="https://colab.research.google.com/github/UmymaM/DeepFake-Detection/blob/main/src/dual_branch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import zipfile as zf
with zf.ZipFile("/content/drive/MyDrive/archive (1).zip","r") as zip_ref:
  zip_ref.extractall("/content")

In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from torchvision.models import EfficientNet_B4_Weights
from tqdm import tqdm

In [4]:
import sys
sys.path.append("/content/drive/MyDrive/DeepFake-Detection/src")

In [5]:
from data.dataset import DeepFakeDataset
from data.transforms import get_train_transform, get_val_transforms
from models.dual_branch import DualBranchNet

In [6]:
# loading pre-split metatdata csvs
train_df=pd.read_csv("/content/drive/MyDrive/train_metadata.csv")
test_df=pd.read_csv("/content/drive/MyDrive/test_metadata.csv")
val_df=pd.read_csv("/content/drive/MyDrive/val_metadata.csv")

In [7]:
# creating 3 dataset objects
train_dataset=DeepFakeDataset(train_df,transform=get_train_transform(),return_fft=True)
val_dataset=DeepFakeDataset(val_df,transform=get_val_transforms(),return_fft=True)
test_dataset=DeepFakeDataset(test_df,transform=get_val_transforms(),return_fft=True)

In [8]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,num_workers=2,pin_memory=True)
val_loader=DataLoader(val_dataset,batch_size=32,shuffle=False,num_workers=2,pin_memory=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False,num_workers=2,pin_memory=True)

In [9]:
rgb,fft,labels=next(iter(train_loader))
print(f"RGB shape:    {rgb.shape}\nFFT shape:    {fft.shape}\nLabels shape: {labels.shape}")

RGB shape:    torch.Size([32, 3, 224, 224])
FFT shape:    torch.Size([32, 1, 224, 224])
Labels shape: torch.Size([32])


In [10]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

Using Device: cuda


In [11]:
import inspect
from data.dataset import DeepFakeDataset
print(inspect.getsource(DeepFakeDataset))

class DeepFakeDataset(Dataset):
  def __init__(self,df,transform=None,return_fft=False):
    self.df=df
    self.transform=transform
    self.return_fft=return_fft

  def __len__(self):
    return len(self.df)

  def _compute_fft(self, image_tensor):
    # convert RGB tensor to grayscale
    gray=0.299*image_tensor[0]+0.587*image_tensor[1]+0.114*image_tensor[2]
    fft=torch.fft.fft2(gray)
    fft_shift=torch.fft.fftshift(fft)
    magnitude=torch.log(torch.abs(fft_shift)+1e-8)
    magnitude=(magnitude-magnitude.min())/(magnitude.max()-magnitude.min()+1e-8)
    return magnitude.unsqueeze(0)
  
  def __getitem__(self,idx):
    img_path=self.df.iloc[idx,0]
    label=self.df.iloc[idx,1]
    image=Image.open(img_path).convert("RGB")
    if self.transform:
      image=self.transform(image)
    if self.return_fft:
      fft_spectrum = self._compute_fft(image)
      return image, fft_spectrum, torch.tensor(label, dtype=torch.float32)
    return image,torch.tensor(label, dtype=torch.float32)



In [12]:
# creating a new model w random weights+moving it to gpu
model=DualBranchNet().to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 187MB/s]


In [13]:
# loading the saved checkpoint from drive
checkpoint=torch.load("/content/drive/MyDrive/dual_branch_latest.pth", map_location=device)
# loads the saved weights and replaces random initialization
# w trained wrights
model.load_state_dict(checkpoint["model_state_dict"])
best_val_loss=checkpoint["best_val_loss"]
history=checkpoint["history"]
print(f"Resumed from epoch {checkpoint['epoch'] + 1}")
print(f"Best val loss so far: {best_val_loss:.5f}")

Resumed from epoch 6
Best val loss so far: 0.16876


In [13]:
# baseline_weights = torch.load("/content/drive/MyDrive/best_model.pth", map_location=device)
# spatial_weights  = {k.replace("features.", ""): v
#     for k, v in baseline_weights.items()
#     if k.startswith("features.")
# }

In [14]:
# model.spatial_branch.load_state_dict(spatial_weights,strict=True)

<All keys matched successfully>

In [14]:
# binary cross entropy with logits loss
criterion=nn.BCEWithLogitsLoss()

In [15]:
def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
    # switching to training mode
    running_loss,all_preds,all_labels=0.0,[],[]
    # initializing empty lists
    for rgb,fft,labels in tqdm(train_loader,desc='Training'):
      # iterating through each batch in the loader
        rgb,fft,labels=rgb.to(device),fft.to(device),labels.to(device).unsqueeze(1)
        # moving tensors to gpu
        optimizer.zero_grad()
        # clears gradients from prev batch
        outputs=model(rgb,fft)
        # forward pass: rgb goes thru spatial branch and fft thru freq branch
        loss=criterion(outputs,labels)
        # computing loss
        loss.backward()
        # backpropagation
        optimizer.step()
        # updates parameters (learning rate+weights)
        running_loss+=loss.item()
        preds=torch.sigmoid(outputs).round().detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
    return running_loss/len(train_loader),accuracy_score(all_labels, all_preds)

In [16]:
def validate(model, loader, criterion, device):
    model.eval()
    running_loss,all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for rgb, fft, labels in tqdm(loader, desc="Validation"):
            rgb=rgb.to(device)
            fft=fft.to(device)
            labels=labels.to(device).unsqueeze(1)
            outputs=model(rgb,fft)
            loss=criterion(outputs, labels)
            running_loss += loss.item()
            preds = torch.sigmoid(outputs).round().detach().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    return running_loss/len(loader),accuracy_score(all_labels, all_preds)

In [25]:
# method to compute area under the ROC curve
from sklearn.metrics import roc_auc_score
def get_auc(model,loader,device):
  model.eval()
  # switching to evaluation mode(disbles dropout+batchnorm)
  all_probs,all_labels=[],[]
  with torch.no_grad():
    # disabling gradient computation
    for batch in tqdm(loader,desc=f"AUC Evaluation"):
      # iterating thru every batch in the dataloader
      # using tqdm to show a progress bar
      rgb,fft,labels=batch
      rgb,fft,labels=rgb.to(device),fft.to(device),labels.to(device).unsqueeze(1)
      # shifting the 3 tensors to gpu
      outputs=model(rgb,fft)
      probs=torch.sigmoid(outputs).cpu().numpy()
      # sigmoid converts raw logits to probabilites (1-0)
      all_labels.extend(labels.cpu().numpy())
      # shofting labels to cpu and converting them into an np array
      all_probs.extend(probs)
  return roc_auc_score(all_labels,all_probs)


In [18]:
# for param in model.spatial_branch.parameters():
#   param.requires_grad=False

In [19]:
# optimizer_s1=optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
#     lr=1e-3, weight_decay=1e-2
# )

In [20]:
# scheduler_s1=optim.lr_scheduler.CosineAnnealingLR(optimizer_s1, T_max=4)

In [21]:
# best_val_loss=float("inf")
# history={"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

In [22]:
# for epoch in range(4):
#     print(f"Epoch {epoch+1}/4")
#     train_loss,train_acc=train_one_epoch(model, train_loader, optimizer_s1, criterion, device)
#     val_loss,val_acc=validate(model, val_loader, criterion, device)
#     scheduler_s1.step()
#     history["train_loss"].append(train_loss)
#     history["val_loss"].append(val_loss)
#     history["train_acc"].append(train_acc)
#     history["val_acc"].append(val_acc)
#     print(f"Train Loss: {train_loss:.5f} | Train Acc: {train_acc:.5f}")
#     print(f"Val Loss:   {val_loss:.5f} | Val Acc:   {val_acc:.5f}")
#     if val_loss < best_val_loss:
#         best_val_loss = val_loss
#         torch.save(model.state_dict(),"/content/drive/MyDrive/dual_branch_best.pth")
#         print("Best model saved ✓")
#     torch.save({
#         "epoch": epoch,
#         "model_state_dict": model.state_dict(),
#         "optimizer_state_dict": optimizer_s1.state_dict(),
#         "best_val_loss": best_val_loss,
#         "history": history
#     }, "/content/drive/MyDrive/dual_branch_latest.pth")


Epoch 1/4


Validation: 100%|██████████| 938/938 [02:48<00:00,  5.55it/s]


Train Loss: 0.35216 | Train Acc: 0.83581
Val Loss:   0.23656 | Val Acc:   0.88773
Best model saved ✓
Epoch 2/4


Validation: 100%|██████████| 938/938 [02:39<00:00,  5.89it/s]


Train Loss: 0.34323 | Train Acc: 0.83697
Val Loss:   0.23305 | Val Acc:   0.89157
Best model saved ✓
Epoch 3/4


Validation: 100%|██████████| 938/938 [02:39<00:00,  5.90it/s]


Train Loss: 0.33789 | Train Acc: 0.83856
Val Loss:   0.23286 | Val Acc:   0.88450
Best model saved ✓
Epoch 4/4


Training:  77%|███████▋  | 3367/4375 [25:54<07:45,  2.17it/s]


KeyboardInterrupt: 

In [18]:
# unfreezing spatial features
for param in model.spatial_branch.parameters():
    param.requires_grad=True

In [19]:
optimizer_s2=optim.AdamW(model.parameters(),lr=1e-5,weight_decay=1e-2)
scheduler_s2=optim.lr_scheduler.CosineAnnealingLR(optimizer_s2,T_max=4)

In [23]:
for epoch in range(9,12):
    print(f"Epoch {epoch+1}/12")
    train_loss,train_acc=train_one_epoch(model, train_loader, optimizer_s2, criterion, device)
    val_loss,val_acc=validate(model, val_loader, criterion, device)
    scheduler_s2.step()
    # updating learning rate after each epoch
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    print(f"Train Loss: {train_loss:.5f} | Train Acc: {train_acc:.5f}")
    print(f"Val Loss:   {val_loss:.5f} | Val Acc:   {val_acc:.5f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "/content/drive/MyDrive/dual_branch_best.pth")
        print("Best model saved ✓")
    torch.save({
        "epoch": epoch + 4,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer_s2.state_dict(),
        "best_val_loss": best_val_loss,
        "history": history
    }, "/content/drive/MyDrive/dual_branch_latest.pth")


Epoch 10/12


Validation: 100%|██████████| 938/938 [02:48<00:00,  5.58it/s]


Train Loss: 0.16900 | Train Acc: 0.91767
Val Loss:   0.13542 | Val Acc:   0.92827
Best model saved ✓
Epoch 11/12


Validation: 100%|██████████| 938/938 [02:46<00:00,  5.62it/s]


Train Loss: 0.16929 | Train Acc: 0.91742
Val Loss:   0.13741 | Val Acc:   0.92773
Epoch 12/12


Training:  20%|█▉        | 867/4375 [08:31<34:29,  1.70it/s]


KeyboardInterrupt: 

In [26]:
dual_auc=get_auc(model,val_loader,device)
print(f"Dual Branch AUC: {dual_auc}")

AUC Evaluation: 100%|██████████| 938/938 [02:42<00:00,  5.76it/s]


Dual Branch AUC: 0.986846117912999
